<a href="https://colab.research.google.com/github/dbro-10/first_machine_learning_for_heart_disease/blob/main/Own_attempt_at_creating_ML_with_logisitical_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing data from libaries  

In [2]:
!pip install shap kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("redwankarimsony/heart-disease-data")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'heart-disease-data' dataset.
Path to dataset files: /kaggle/input/heart-disease-data


In [3]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## Data collection and processing

In [4]:
import os
print(os.listdir(path))

['heart_disease_uci.csv']


In [5]:
# loading the csv data to a pandas dataframe
heart_data = pd.read_csv(f'{path}/heart_disease_uci.csv')

In [6]:
#adding first 5
heart_data.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [7]:
#adding 5 last (num reperesents the predicted attribute for heart disease)
heart_data.tail()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
915,916,54,Female,VA Long Beach,asymptomatic,127.0,333.0,True,st-t abnormality,154.0,False,0.0,NaN,NaN,NaN,1
916,917,62,Male,VA Long Beach,typical angina,NaN,139.0,False,st-t abnormality,NaN,NaN,NaN,NaN,NaN,NaN,0
917,918,55,Male,VA Long Beach,asymptomatic,122.0,223.0,True,st-t abnormality,100.0,False,0.0,NaN,NaN,fixed defect,2
918,919,58,Male,VA Long Beach,asymptomatic,NaN,385.0,True,lv hypertrophy,NaN,NaN,NaN,NaN,NaN,NaN,0
919,920,62,Male,VA Long Beach,atypical angina,120.0,254.0,False,lv hypertrophy,93.0,True,0.0,NaN,NaN,NaN,1


In [8]:
# gettinf info about the data
heart_data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        920 non-null    int64  
 1   age       920 non-null    int64  
 2   sex       920 non-null    object 
 3   dataset   920 non-null    object 
 4   cp        920 non-null    object 
 5   trestbps  861 non-null    float64
 6   chol      890 non-null    float64
 7   fbs       830 non-null    object 
 8   restecg   918 non-null    object 
 9   thalch    865 non-null    float64
 10  exang     865 non-null    object 
 11  oldpeak   858 non-null    float64
 12  slope     611 non-null    object 
 13  ca        309 non-null    float64
 14  thal      434 non-null    object 
 15  num       920 non-null    int64  
dtypes: float64(5), int64(3), object(8)
memory usage: 115.1+ KB


In [9]:
# Calculate the total number of missing values in the DataFrame
total_missing_values = heart_data.isnull().sum().sum()
print(f"Total missing values in the DataFrame: {total_missing_values}")

Total missing values in the DataFrame: 1759


In [10]:
#checking for missing values
# heart_data has 920 rows and 16 columns = 14720 data values
heart_data.isnull().sum()

,0
id,0
age,0
sex,0
dataset,0
cp,0
trestbps,59
chol,30
fbs,90
restecg,2
thalch,55


In [11]:
# stattistical measures about the data
heart_data.describe()

,id,age,trestbps,chol,thalch,oldpeak,ca,num
count,920.000000,920.000000,861.000000,890.000000,865.000000,858.000000,309.000000,920.000000
mean,460.500000,53.510870,132.132404,199.130337,137.545665,0.878788,0.676375,0.995652
std,265.725422,9.424685,19.066070,110.780810,25.926276,1.091226,0.935653,1.142693
min,1.000000,28.000000,0.000000,0.000000,60.000000,-2.600000,0.000000,0.000000
25%,230.750000,47.000000,120.000000,175.000000,120.000000,0.000000,0.000000,0.000000
50%,460.500000,54.000000,130.000000,223.000000,140.000000,0.500000,0.000000,1.000000
75%,690.250000,60.000000,140.000000,268.000000,157.000000,1.500000,1.000000,2.000000
max,920.000000,77.000000,200.000000,603.000000,202.000000,6.200000,3.000000,4.000000


In [12]:
# checking the distrubution of the target variable
heart_data['num'].value_counts()

,count
num,
0,411
1,265
2,109
3,107
4,28


num is the target variable. 0 = no heart disease

1, 2, 3, 4 = increasing severity of heart disease

splitting the features and targets

In [13]:
# going to analyse the columns to predict chance of heart disease or not
X = heart_data.drop(columns='num', axis=1)
Y = heart_data['num']

# Convert categorical columns to numerical using one-hot encoding
X = pd.get_dummies(X, columns=['sex', 'dataset', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal'], drop_first=True)

# Combine X and Y to drop rows with NaNs consistently across both
df_combined = pd.concat([X, Y], axis=1)
df_combined = df_combined.dropna()

X = df_combined.drop(columns='num', axis=1)
Y = df_combined['num']

In [14]:
print(X.head())

   id  age  trestbps   chol  thalch  oldpeak   ca  sex_Male  dataset_Hungary  \
0   1   63     145.0  233.0   150.0      2.3  0.0      True            False   
1   2   67     160.0  286.0   108.0      1.5  3.0      True            False   
2   3   67     120.0  229.0   129.0      2.6  2.0      True            False   
3   4   37     130.0  250.0   187.0      3.5  0.0      True            False   
4   5   41     130.0  204.0   172.0      1.4  0.0     False            False   

   dataset_Switzerland  ...  cp_non-anginal  cp_typical angina  fbs_True  \
0                False  ...           False               True      True   
1                False  ...           False              False     False   
2                False  ...           False              False     False   
3                False  ...            True              False     False   
4                False  ...           False              False     False   

   restecg_normal  restecg_st-t abnormality  exang_True  slope

In [15]:
print(Y)

0      0
1      2
2      1
3      0
4      0
      ..
676    1
691    1
717    0
748    1
759    0
Name: num, Length: 308, dtype: int64


Splitting the data into training data and test data

In [16]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=2)

In [17]:
print(X.shape, X_train.shape, X_test.shape)

(308, 22) (246, 22) (62, 22)


Model training

Logisitic regression model

In [18]:
model = LogisticRegression()

In [19]:
# Training the logisiticRegression model with training data
model.fit(X_train, Y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

Model Evalutaion

Accuracy scores

In [20]:
# accuracy on training data
x_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(x_train_prediction, Y_train)

In [21]:
print ('Accuracy on training data:', training_data_accuracy)

Accuracy on training data: 0.5934959349593496


As you can see the accuracy is very low meaning our model does not have a good prediction rate for Heart disease. this is because of the use of the logisitcal regression model and poorly inputed data.

In [22]:
# accuracy on test data
x_train_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(x_train_prediction, Y_test)

In [23]:
print('Accuracy on test data:', test_data_accuracy)

Accuracy on test data: 0.5806451612903226


Again very low percentage because of overfitting the model with the data

# Building predicitive system

In [26]:
# Example input: [id, age, trestbps, chol, thalch, oldpeak, ca, sex_Male, dataset_Hungary, ...]
# (Match the order and encoded columns from X.columns)
input_data = [1, 63, 145, 233, 150, 2.3, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0]  # Example row
input_data_as_numpy_array = np.asarray(input_data)
input_data_reshaped = input_data_as_numpy_array.reshape(1, -1)
prediction = model.predict(input_data_reshaped)
print(prediction)

if prediction[0] == 0:
    print('The Person does not have Heart Disease')
else:
    print('The Person has Heart Disease (severity:', prediction[0], ')')

[0]
The Person does not have Heart Disease


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
